# PawHealthAI - Vitality Score Engine v0.2
### Validation-ready, non-diagnostic scoring engine & test harness

This notebook upgrades the early prototype into a **validation-ready test harness**
that PawHealthAI, veterinarians and engineering can review and critique.

**What changed from the prototype (v0.1):**
- Proper dataclasses for every entity (`PetProfile`, `CheckInData`, `BaselineSummary`,
  `RedFlagResult`, `VitalityScoreResult`) - no references to undefined fields.
- Scoring rebuilt around **comparison with the pet's own recent baseline** rather than
  fixed deductions, with an honest *building-baseline* state when data is thin.
- Red-flag safety overrides moved into a **separate layer** with three tiers
  (emergency / urgent / monitor). Acute signs never just "lower a score".
- **All diagnostic disease outputs removed.** The engine never names a disease.
- A synthetic dataset (>= 50 check-ins) and an expected-output table (>= 10 scenarios)
  so the logic can be evaluated, not just executed.

> **Safety notice.** This engine is *provisional* and *not clinically validated*. It does
> not diagnose, and it does not claim to know whether a pet is healthy or sick. Every
> threshold is a placeholder pending veterinary review. See `vitality_score_engine_v0_2_report.md`.

The reusable engine lives in **`vitality_score_engine_v0_2.py`**; this notebook is the
exploratory test harness that imports it, generates data, and checks behaviour.

## 1. Import the engine module
The scoring logic is kept in a separate, dependency-free module so it can be reused and reviewed independently of these exploratory cells.

In [1]:
import dataclasses
from datetime import datetime, date, timedelta

import pandas as pd

from vitality_score_engine_v0_2 import (
    PetProfile,
    CheckInData,
    BaselineSummary,
    RedFlagResult,
    VitalityScoreResult,
    RedFlagEngine,
    VitalityScoreEngine,
    process_checkin,
)

pd.set_option("display.max_colwidth", 90)
print("Engine v0.2 imported. Data models:",
      ", ".join(c.__name__ for c in [PetProfile, CheckInData, BaselineSummary,
                                     RedFlagResult, VitalityScoreResult]))

Engine v0.2 imported. Data models: PetProfile, CheckInData, BaselineSummary, RedFlagResult, VitalityScoreResult


## 2. Data models (recap)
The engine module defines clean dataclasses. Key points:

| Model | Purpose |
|-------|---------|
| `PetProfile` | Static pet info; drives breed-aware weighting. |
| `CheckInData` | One weekly check-in. Wellbeing scores are **1-5**; acute signs are booleans. |
| `BaselineSummary` | The pet's own recent normal (averages of appetite/energy/sleep/activity). |
| `RedFlagResult` | Safe escalation fields (`trigger`, `severity_tier`, `clinical_reason_for_escalation`, `recommended_user_pathway`, `vet_validation_required`) - replaces the old unsafe `potential_diseases`. |
| `VitalityScoreResult` | Product-facing result: `score`, `band`, `baseline_status`, `confidence`, `override`, `explanation`. |

`CheckInData` carries exactly the fields from the brief's check-in schema (section 4.2):

In [2]:
# Show the CheckInData schema (field name -> type) straight from the dataclass.
for f in dataclasses.fields(CheckInData):
    print(f"{f.name:28s} {getattr(f.type, '__name__', f.type)}")

pet_id                       str
timestamp                    datetime
appetite_score               int
energy_score                 int
sleep_quality_score          int
activity_minutes             int
toileting_status             str
water_intake_status          str
blood_in_stool               bool
vomiting                     bool
diarrhoea                    bool
breathing_difficulty         bool
collapse                     bool
seizure                      bool
suspected_toxin_ingestion    bool
unable_to_urinate            bool
severe_bleeding              bool
tick_found                   bool
pain_or_discomfort_signs     bool
weight_kg                    Optional[float]
owner_concern_level          Optional[int]


## 3. Synthetic scenarios & dataset
We build **11 named scenarios** spanning normal, mild, concerning and emergency cases.
Each scenario provides a `PetProfile`, a recent check-in `history`, and a `current`
check-in to score. All check-ins are flattened to a CSV (`synthetic_checkins_v0_2.csv`)
with **>= 50 rows**.

In [3]:
WEEK0 = datetime(2026, 1, 1)


def ci(pet_id, week, appetite=4, energy=4, sleep=4, activity=60, **kw):
    """Convenience builder for a CheckInData at a given week offset."""
    return CheckInData(
        pet_id=pet_id, timestamp=WEEK0 + timedelta(days=7 * week),
        appetite_score=appetite, energy_score=energy,
        sleep_quality_score=sleep, activity_minutes=activity, **kw,
    )


def normal_history(pet_id, n=5, appetite=4, energy=4, sleep=4, activity=60):
    """A run of `n` steady, healthy weekly check-ins."""
    return [ci(pet_id, w, appetite, energy, sleep, activity) for w in range(n)]


SCENARIOS = []


def scenario(name, pet, history, current, expected_bands, expected_tier, note):
    SCENARIOS.append(dict(name=name, pet=pet, history=history, current=current,
                          expected_bands=expected_bands, expected_tier=expected_tier, note=note))


# 1. Stable adult dog -> high/stable, no red flags
p = PetProfile("p01", "Bella", "dog", "Labrador", date(2019, 5, 1))
scenario("stable_adult_dog", p, normal_history("p01"),
         ci("p01", 5, 4, 4, 4, 60),
         ["Bright Green", "Medium Green"], None,
         "Normal appetite/energy/sleep/toileting -> high & stable, no red flags.")

# 2. High-energy breed with reduced activity -> Watch
p = PetProfile("p02", "Rex", "dog", "Vizsla", date(2021, 3, 1), is_high_energy=True)
scenario("high_energy_reduced_activity", p, normal_history("p02", 5, 4, 4, 4, 90),
         ci("p02", 5, appetite=4, energy=3, sleep=4, activity=25),
         ["Watch", "Medium Green"], None,
         "High-energy dog, activity & energy down vs baseline -> Watch, monitor.")

# 3. Senior dog with gradual decline -> Watch
p = PetProfile("p03", "Max", "dog", "Golden Retriever", date(2012, 1, 1))
hist = [ci("p03", 0, 4, 4, 4, 50), ci("p03", 1, 4, 3, 4, 45), ci("p03", 2, 3, 3, 3, 40),
        ci("p03", 3, 3, 3, 3, 35), ci("p03", 4, 3, 3, 3, 30)]
scenario("senior_gradual_decline", p, hist,
         ci("p03", 5, appetite=3, energy=2, sleep=3, activity=25),
         ["Watch", "Medium Green"], None,
         "Senior dog slow decline -> Watch band, no acute override.")

# 4. Puppy with variable sleep and activity -> stable within own range
p = PetProfile("p04", "Luna", "dog", "Beagle", date(2025, 9, 1))
hist = [ci("p04", 0, 4, 5, 3, 80), ci("p04", 1, 5, 4, 5, 40), ci("p04", 2, 4, 5, 2, 90),
        ci("p04", 3, 5, 4, 4, 50), ci("p04", 4, 4, 5, 3, 70)]
scenario("puppy_variable_sleep_activity", p, hist,
         ci("p04", 5, appetite=5, energy=5, sleep=3, activity=75),
         ["Bright Green", "Medium Green"], None,
         "Puppy naturally variable but within own range -> stable.")

# 5. Cat with a mild appetite drop -> small decrease, monitor if persists
p = PetProfile("p05", "Milo", "cat", "Domestic Shorthair", date(2020, 6, 1))
scenario("cat_appetite_drop", p, normal_history("p05", 5, 4, 4, 4, 30),
         ci("p05", 5, appetite=3, energy=4, sleep=4, activity=28),
         ["Medium Green", "Watch"], None,
         "Single-day mild appetite drop, no other signs -> small decrease, monitor if persists.")

# 6. Dog with blood in stool -> urgent override
p = PetProfile("p06", "Daisy", "dog", "Cocker Spaniel", date(2018, 4, 1))
scenario("dog_blood_in_stool", p, normal_history("p06"),
         ci("p06", 5, appetite=3, energy=2, sleep=3, activity=40, blood_in_stool=True),
         ["Action Needed"], "urgent",
         "Blood in stool -> urgent override, Action Needed pathway.")

# 7. Pet with increased thirst across multiple check-ins -> urgent override
p = PetProfile("p07", "Coco", "dog", "Poodle", date(2016, 2, 1))
hist = normal_history("p07", 4)
hist.append(ci("p07", 4, water_intake_status="increased"))
scenario("increased_thirst_repeated", p, hist,
         ci("p07", 5, appetite=3, energy=3, sleep=4, activity=50, water_intake_status="increased"),
         ["Action Needed"], "urgent",
         "Increased thirst over repeated check-ins -> urgent override (non-diagnostic wording).")

# 8. Pet with persistent low appetite and lethargy -> urgent override
p = PetProfile("p08", "Buddy", "dog", "Boxer", date(2017, 7, 1))
hist = normal_history("p08", 4)
hist.append(ci("p08", 4, appetite=2, energy=2))
scenario("low_appetite_lethargy", p, hist,
         ci("p08", 5, appetite=2, energy=2, sleep=3, activity=30),
         ["Action Needed"], "urgent",
         "Persistent low appetite + lethargy -> urgent override.")

# 9. Emergency red-flag case (collapse) -> emergency override
p = PetProfile("p09", "Shadow", "dog", "German Shepherd", date(2015, 8, 1))
scenario("emergency_collapse", p, normal_history("p09"),
         ci("p09", 5, appetite=2, energy=1, sleep=2, activity=0, collapse=True),
         ["Action Needed"], "emergency",
         "Collapse -> emergency pathway, no reassuring score.")

# 10. Inconsistent check-in history -> low-confidence, baseline-building
p = PetProfile("p10", "Ziggy", "dog", "Border Collie", date(2022, 11, 1), is_high_energy=True)
scenario("inconsistent_low_confidence", p, normal_history("p10", 2),
         ci("p10", 3, appetite=3, energy=3, sleep=3, activity=40),
         ["Building Baseline"], None,
         "Fewer than 4 check-ins -> baseline-building state, low confidence.")

# 11. Single vomiting -> monitor advisory, score still shown
p = PetProfile("p11", "Rosie", "dog", "Dachshund", date(2019, 12, 1))
scenario("single_vomiting_monitor", p, normal_history("p11"),
         ci("p11", 5, appetite=4, energy=4, sleep=4, activity=55, vomiting=True),
         ["Bright Green", "Medium Green", "Watch"], "monitor",
         "Single vomiting -> monitor advisory attached, wellbeing score still shown.")

print(f"{len(SCENARIOS)} scenarios defined.")

11 scenarios defined.


In [4]:
# Flatten every check-in (history + current) into one tidy CSV.
rows = []
for s in SCENARIOS:
    for role, checkin in (
        [("history", h) for h in s["history"]] + [("current", s["current"])]
    ):
        row = {"scenario": s["name"], "pet_id": s["pet"].pet_id,
               "pet_name": s["pet"].name, "role": role}
        row.update(dataclasses.asdict(checkin))
        rows.append(row)

checkins_df = pd.DataFrame(rows)
checkins_df.to_csv("synthetic_checkins_v0_2.csv", index=False)
print(f"Saved synthetic_checkins_v0_2.csv with {len(checkins_df)} check-ins "
      f"across {checkins_df['scenario'].nunique()} scenarios.")
checkins_df.head(8)

Saved synthetic_checkins_v0_2.csv with 63 check-ins across 11 scenarios.


,scenario,pet_id,pet_name,role,timestamp,appetite_score,energy_score,sleep_quality_score,activity_minutes,toileting_status,...,breathing_difficulty,collapse,seizure,suspected_toxin_ingestion,unable_to_urinate,severe_bleeding,tick_found,pain_or_discomfort_signs,weight_kg,owner_concern_level
0,stable_adult_dog,p01,Bella,history,2026-01-01,4,4,4,60,normal,...,False,False,False,False,False,False,False,False,None,None
1,stable_adult_dog,p01,Bella,history,2026-01-08,4,4,4,60,normal,...,False,False,False,False,False,False,False,False,None,None
2,stable_adult_dog,p01,Bella,history,2026-01-15,4,4,4,60,normal,...,False,False,False,False,False,False,False,False,None,None
3,stable_adult_dog,p01,Bella,history,2026-01-22,4,4,4,60,normal,...,False,False,False,False,False,False,False,False,None,None
4,stable_adult_dog,p01,Bella,history,2026-01-29,4,4,4,60,normal,...,False,False,False,False,False,False,False,False,None,None
5,stable_adult_dog,p01,Bella,current,2026-02-05,4,4,4,60,normal,...,False,False,False,False,False,False,False,False,None,None
6,high_energy_reduced_activity,p02,Rex,history,2026-01-01,4,4,4,90,normal,...,False,False,False,False,False,False,False,False,None,None
7,high_energy_reduced_activity,p02,Rex,history,2026-01-08,4,4,4,90,normal,...,False,False,False,False,False,False,False,False,None,None


## 4. Expected outputs table
For each scenario we declare the **expected behaviour before running the engine**.
This is what turns "code that runs" into "logic that can be evaluated". Saved to
`expected_outputs_v0_2.csv`.

In [5]:
expected_df = pd.DataFrame([{
    "scenario": s["name"],
    "input_pattern": s["note"],
    "expected_band(s)": " / ".join(s["expected_bands"]),
    "expected_override": s["expected_tier"] or "none",
    "expected_explanation_style": "safe, non-diagnostic, vet review if persists/worsens",
} for s in SCENARIOS])

expected_df.to_csv("expected_outputs_v0_2.csv", index=False)
expected_df

,scenario,input_pattern,expected_band(s),expected_override,expected_explanation_style
0,stable_adult_dog,"Normal appetite/energy/sleep/toileting -> high & stable, no red flags.",Bright Green / Medium Green,none,"safe, non-diagnostic, vet review if persists/worsens"
1,high_energy_reduced_activity,"High-energy dog, activity & energy down vs baseline -> Watch, monitor.",Watch / Medium Green,none,"safe, non-diagnostic, vet review if persists/worsens"
2,senior_gradual_decline,"Senior dog slow decline -> Watch band, no acute override.",Watch / Medium Green,none,"safe, non-diagnostic, vet review if persists/worsens"
3,puppy_variable_sleep_activity,Puppy naturally variable but within own range -> stable.,Bright Green / Medium Green,none,"safe, non-diagnostic, vet review if persists/worsens"
4,cat_appetite_drop,"Single-day mild appetite drop, no other signs -> small decrease, monitor if persists.",Medium Green / Watch,none,"safe, non-diagnostic, vet review if persists/worsens"
5,dog_blood_in_stool,"Blood in stool -> urgent override, Action Needed pathway.",Action Needed,urgent,"safe, non-diagnostic, vet review if persists/worsens"
6,increased_thirst_repeated,Increased thirst over repeated check-ins -> urgent override (non-diagnostic wording).,Action Needed,urgent,"safe, non-diagnostic, vet review if persists/worsens"
7,low_appetite_lethargy,Persistent low appetite + lethargy -> urgent override.,Action Needed,urgent,"safe, non-diagnostic, vet review if persists/worsens"
8,emergency_collapse,"Collapse -> emergency pathway, no reassuring score.",Action Needed,emergency,"safe, non-diagnostic, vet review if persists/worsens"
9,inconsistent_low_confidence,"Fewer than 4 check-ins -> baseline-building state, low confidence.",Building Baseline,none,"safe, non-diagnostic, vet review if persists/worsens"


## 5. Run the engine and compare actual vs expected
We run the full pipeline (`process_checkin`) for every scenario and check that the
**band** and **override tier** match expectations. Each result also exposes
score, baseline status, confidence, override status and a plain-English explanation.

In [6]:
results = []
for s in SCENARIOS:
    res = process_checkin(s["pet"], s["history"], s["current"])
    tier = res.override.severity_tier if res.override else None
    band_ok = res.band in s["expected_bands"]
    tier_ok = tier == s["expected_tier"]
    results.append({
        "scenario": s["name"],
        "score": res.score,
        "band": res.band,
        "baseline_status": res.baseline_status,
        "confidence": res.confidence,
        "override_tier": tier or "none",
        "pass": "PASS" if (band_ok and tier_ok) else "FAIL",
        "explanation": res.explanation,
    })

results_df = pd.DataFrame(results)
results_df.to_csv("actual_outputs_v0_2.csv", index=False)
results_df[["scenario", "score", "band", "baseline_status", "confidence",
            "override_tier", "pass"]]

,scenario,score,band,baseline_status,confidence,override_tier,pass
0,stable_adult_dog,100.0,Bright Green,established,moderate,none,PASS
1,high_energy_reduced_activity,54.0,Watch,established,moderate,none,PASS
2,senior_gradual_decline,50.0,Watch,established,moderate,none,PASS
3,puppy_variable_sleep_activity,95.0,Bright Green,established,moderate,none,PASS
4,cat_appetite_drop,81.0,Medium Green,established,moderate,none,PASS
5,dog_blood_in_stool,NaN,Action Needed,override,high,urgent,PASS
6,increased_thirst_repeated,NaN,Action Needed,override,high,urgent,PASS
7,low_appetite_lethargy,NaN,Action Needed,override,high,urgent,PASS
8,emergency_collapse,NaN,Action Needed,override,high,emergency,PASS
9,inconsistent_low_confidence,NaN,Building Baseline,building,low,none,PASS


In [7]:
n_pass = (results_df["pass"] == "PASS").sum()
print(f"{n_pass}/{len(results_df)} scenarios passed.")
assert n_pass == len(results_df), "Some scenarios did not match expected behaviour!"
print("All scenarios match their expected band and override tier.")

11/11 scenarios passed.
All scenarios match their expected band and override tier.


## 6. Worked example outputs
A closer look at representative results - note the safe, non-diagnostic language and that emergency/urgent cases withhold a reassuring score.

In [8]:
def show(scenario_name):
    s = next(x for x in SCENARIOS if x["name"] == scenario_name)
    res = process_checkin(s["pet"], s["history"], s["current"])
    print(f"=== {s['name']}  ({s['pet'].name}, {s['pet'].breed}) ===")
    print(f"score            : {res.score}")
    print(f"band             : {res.band}")
    print(f"baseline_status  : {res.baseline_status}")
    print(f"confidence       : {res.confidence}")
    if res.override and res.override.triggered:
        print(f"override         : {res.override.severity_tier} "
              f"(trigger={res.override.trigger}, vet_validation_required="
              f"{res.override.vet_validation_required})")
    else:
        print("override         : none")
    print(f"drivers          : {res.drivers}")
    print(f"explanation      : {res.explanation}")
    print()


for name in ["stable_adult_dog", "cat_appetite_drop", "dog_blood_in_stool",
             "increased_thirst_repeated", "emergency_collapse", "inconsistent_low_confidence"]:
    show(name)

=== stable_adult_dog  (Bella, Labrador) ===
score            : 100
band             : Bright Green
baseline_status  : established
confidence       : moderate
override         : none
drivers          : ['all tracked indicators in line with usual baseline']
explanation      : Bella's indicators are tracking in line with their usual baseline this week. No emergency red flags were reported. Keep up the regular check-ins.

=== cat_appetite_drop  (Milo, Domestic Shorthair) ===
score            : 81
band             : Medium Green
baseline_status  : established
confidence       : moderate
override         : none
drivers          : ['appetite below usual']
explanation      : Milo's indicators are tracking in line with their usual baseline this week. No emergency red flags were reported. Keep up the regular check-ins.

=== dog_blood_in_stool  (Daisy, Cocker Spaniel) ===
score            : None
band             : Action Needed
baseline_status  : override
confidence       : high
override         

## 7. Safety check - no disease names leak to users
A guard test asserting the product-facing explanation never contains a diagnostic
disease term (section 4.5 / 7).

In [9]:
BANNED_TERMS = ["diabetes", "kidney disease", "cushing", "gastroenteritis",
                "toxicity", "tick-borne", "cancer", "pancreatitis", "parvo", "disease"]

leaks = []
for s in SCENARIOS:
    res = process_checkin(s["pet"], s["history"], s["current"])
    text = res.explanation.lower()
    for term in BANNED_TERMS:
        if term in text:
            leaks.append((s["name"], term))

assert not leaks, f"Disease term leaked into user-facing output: {leaks}"
print("PASS - no diagnostic disease names appear in any product-facing explanation.")

PASS - no diagnostic disease names appear in any product-facing explanation.


## 8. Deliverables produced by this notebook
| File | Description |
|------|-------------|
| `data_pipeline_vitality_score.ipynb` | This runnable test-harness notebook. |
| `vitality_score_engine_v0_2.py` | Reusable, dependency-free scoring engine. |
| `synthetic_checkins_v0_2.csv` | >= 50 synthetic check-ins across 11 scenarios. |
| `expected_outputs_v0_2.csv` | Scenario-level expected band, override and explanation style. |
| `actual_outputs_v0_2.csv` | Engine output for each scenario (regenerated on every run). |
| `vitality_score_engine_v0_2_report.md` | Technical report: logic, assumptions, limitations, vet-review questions. |

**Reminder:** every threshold and band here is *provisional* and must be reviewed by a
veterinarian before any product use. The engine is a transparent rules harness, not a
diagnostic tool.